In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType
)

print("Real-time streaming notebook ready!")

Real-time streaming notebook ready!


In [0]:
from pyspark.sql import functions as F

# Historical smart-meter dataset
file_path = "/Volumes/workspace/default/data_s/SM Cleaned Data BR2019.csv"

historical_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

# Select and rename the important columns
historical_df = (
    historical_df
    .select(
        F.col("meter").alias("meter_id"),
        F.col("t_kWh").alias("energy_kwh"),
        F.col("z_Avg Voltage (Volt)").alias("avg_voltage"),
        F.col("z_Avg Current (Amp)").alias("avg_current"),
        F.col("y_Freq (Hz)").alias("frequency")
    )
)

print("Historical data loaded successfully!")
print("Rows:", historical_df.count())

Historical data loaded successfully!
Rows: 2919315


In [0]:
meter_profiles_df = (
    historical_df
    .groupBy("meter_id")
    .agg(
        F.expr("percentile_approx(energy_kwh, 0.5)").alias("baseline_energy_3min"),
        F.expr("percentile_approx(avg_voltage, 0.5)").alias("baseline_voltage"),
        F.expr("percentile_approx(avg_current, 0.5)").alias("baseline_current"),
        F.expr("percentile_approx(frequency, 0.5)").alias("baseline_frequency")
    )
    .orderBy("meter_id")
)

display(meter_profiles_df)

meter_id,baseline_energy_3min,baseline_voltage,baseline_current,baseline_frequency
BR02,0.008,250.78,0.75,50.01
BR03,0.013,247.12,1.3,50.01
BR04,0.014,247.77,1.37,50.01
BR05,0.005,247.4,0.45,50.01
BR06,0.018,251.32,1.82,50.01
BR07,0.004,252.27,0.35,50.01
BR08,0.009,251.59,0.96,50.01
BR09,0.009,255.12,1.2,50.01
BR10,0.011,249.8,0.99,50.01
BR11,0.009,246.63,0.91,50.02


In [0]:
id="x7k2pq"
def register_user(
    meter_id,
    monthly_target_kwh,
    monthly_budget,
    home_type="Apartment",
    occupants=2
):
    meter_id = str(meter_id).strip().upper()

    # Check whether the meter already exists
    existing_profile = (
        meter_profiles_df
        .filter(F.col("meter_id") == meter_id)
        .collect()
    )

    if len(existing_profile) > 0:
        # Existing user: use historical profile
        profile = existing_profile[0].asDict()

        user = {
            "meter_id": meter_id,
            "user_type": "Existing User",
            "home_type": home_type,
            "occupants": occupants,
            "monthly_target_kwh": float(monthly_target_kwh),
            "monthly_budget": float(monthly_budget),
            "baseline_energy_3min": float(profile["baseline_energy_3min"]),
            "baseline_voltage": float(profile["baseline_voltage"]),
            "baseline_current": float(profile["baseline_current"]),
            "baseline_frequency": float(profile["baseline_frequency"])
        }

        print("✅ Existing user found!")
        print(f"Meter ID: {meter_id}")
        print("Historical profile will be used.")

    else:
        # New user: create synthetic profile
        user = {
            "meter_id": meter_id,
            "user_type": "New User",
            "home_type": home_type,
            "occupants": occupants,
            "monthly_target_kwh": float(monthly_target_kwh),
            "monthly_budget": float(monthly_budget),

            # Synthetic starting profile
            "baseline_energy_3min": 0.15,
            "baseline_voltage": 230.0,
            "baseline_current": 4.5,
            "baseline_frequency": 50.0
        }

        print("🆕 New user detected!")
        print(f"Meter ID: {meter_id}")
        print("Synthetic smart-meter profile created.")

    return user

In [0]:
id="k9f3aa"
user1 = register_user(
    meter_id="BR03",
    monthly_target_kwh=200,
    monthly_budget=1500,
    home_type="Apartment",
    occupants=3
)

print(user1)

✅ Existing user found!
Meter ID: BR03
Historical profile will be used.
{'meter_id': 'BR03', 'user_type': 'Existing User', 'home_type': 'Apartment', 'occupants': 3, 'monthly_target_kwh': 200.0, 'monthly_budget': 1500.0, 'baseline_energy_3min': 0.013, 'baseline_voltage': 247.12, 'baseline_current': 1.3, 'baseline_frequency': 50.01}


In [0]:
id="p2v6cd"
user2 = register_user(
    meter_id="BR100",
    monthly_target_kwh=250,
    monthly_budget=1800,
    home_type="House",
    occupants=4
)

print(user2)

🆕 New user detected!
Meter ID: BR100
Synthetic smart-meter profile created.
{'meter_id': 'BR100', 'user_type': 'New User', 'home_type': 'House', 'occupants': 4, 'monthly_target_kwh': 250.0, 'monthly_budget': 1800.0, 'baseline_energy_3min': 0.15, 'baseline_voltage': 230.0, 'baseline_current': 4.5, 'baseline_frequency': 50.0}


In [0]:
import random
from datetime import datetime


def generate_reading(user):

    # Decide whether this reading is normal or abnormal
    is_abnormal = random.random() < 0.05   # 5% chance

    # Demo scaling:
    # Generate a realistic energy value for the fast 2-second simulation.
    # The value is treated as simulated household consumption,
    # not as an actual physical 2-second meter measurement.
    demo_energy_base = user["baseline_energy_3min"] * 5

    if not is_abnormal:

        # Normal household variation
        voltage = random.gauss(
            user["baseline_voltage"],
            2.0
        )

        current = max(
            0.1,
            random.gauss(
                user["baseline_current"],
                user["baseline_current"] * 0.10
            )
        )

        frequency = random.gauss(
            user["baseline_frequency"],
            0.03
        )

        # Energy related to current
        energy_kwh = (
            demo_energy_base
            * (current / max(user["baseline_current"], 0.1))
        )

        # Small random variation
        energy_kwh *= random.uniform(0.90, 1.10)

    else:

        # Simulated abnormal event
        abnormal_type = random.choice([
            "HIGH_USAGE",
            "LOW_VOLTAGE",
            "HIGH_VOLTAGE",
            "LOW_FREQUENCY"
        ])

        voltage = user["baseline_voltage"]
        current = user["baseline_current"]
        frequency = user["baseline_frequency"]

        energy_kwh = demo_energy_base

        if abnormal_type == "HIGH_USAGE":

            current *= random.uniform(2.0, 3.0)
            energy_kwh *= random.uniform(2.0, 3.0)

        elif abnormal_type == "LOW_VOLTAGE":

            voltage = random.uniform(170, 190)

        elif abnormal_type == "HIGH_VOLTAGE":

            voltage = random.uniform(250, 270)

        elif abnormal_type == "LOW_FREQUENCY":

            frequency = random.uniform(45, 48)

        print(f"⚠️ Simulated abnormal event: {abnormal_type}")

    reading = {
        "timestamp": datetime.now(),
        "meter_id": user["meter_id"],
        "energy_kwh": round(max(0, energy_kwh), 4),
        "avg_voltage": round(voltage, 2),
        "avg_current": round(current, 2),
        "frequency": round(frequency, 2),
        "abnormal_event": is_abnormal
    }

    return reading

In [0]:
reading1 = generate_reading(user1)

print("Simulated Smart-Meter Reading")

print("Timestamp :", reading1["timestamp"])
print("Meter ID  :", reading1["meter_id"])
print("Energy    :", reading1["energy_kwh"], "kWh")
print("Voltage   :", reading1["avg_voltage"], "V")
print("Current   :", reading1["avg_current"], "A")
print("Frequency :", reading1["frequency"], "Hz")
print("Abnormal  :", reading1["abnormal_event"])

Simulated Smart-Meter Reading
Timestamp : 2026-09-20 07:10:30.368655
Meter ID  : BR03
Energy    : 0.0604 kWh
Voltage   : 243.23 V
Current   : 1.24 A
Frequency : 50.02 Hz
Abnormal  : False


In [0]:
live_table_name = "workspace.default.smart_energy_live"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {live_table_name} (
    timestamp TIMESTAMP,
    meter_id STRING,
    user_type STRING,
    home_type STRING,
    occupants INT,
    energy_kwh DOUBLE,
    avg_voltage DOUBLE,
    avg_current DOUBLE,
    frequency DOUBLE,
    abnormal_event BOOLEAN
)
USING DELTA
""")

print("✅ Live smart-energy Delta table is ready!")

✅ Live smart-energy Delta table is ready!


In [0]:
def save_live_reading(user):
    
    reading = generate_reading(user)
    
    reading["user_type"] = user["user_type"]
    reading["home_type"] = user["home_type"]
    reading["occupants"] = int(user["occupants"])
    
    live_reading_df = spark.createDataFrame([reading])
    
    live_reading_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(live_table_name)
    
    return reading

In [0]:
reading = save_live_reading(user1)

print("✅ New live reading stored!")
print("--------------------------------")
print("Meter ID :", reading["meter_id"])
print("Time     :", reading["timestamp"])
print("Energy   :", reading["energy_kwh"], "kWh")
print("Voltage  :", reading["avg_voltage"], "V")
print("Current  :", reading["avg_current"], "A")
print("Frequency:", reading["frequency"], "Hz")
print("Abnormal :", reading["abnormal_event"])

✅ New live reading stored!
--------------------------------
Meter ID : BR03
Time     : 2026-09-20 07:10:31.559384
Energy   : 0.0652 kWh
Voltage  : 243.77 V
Current  : 1.4 A
Frequency: 50.0 Hz
Abnormal : False


In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {live_table_name}
        ORDER BY timestamp DESC
        LIMIT 10
    """)
)

timestamp,meter_id,user_type,home_type,occupants,energy_kwh,avg_voltage,avg_current,frequency,abnormal_event
2026-09-20T07:10:31.559Z,BR03,Existing User,Apartment,3,0.0652,243.77,1.4,50.0,false
2026-09-02T17:15:00.000Z,BR03,Existing User,Apartment,3,0.0766,248.86,1.47,50.02,false
2026-09-02T17:15:00.000Z,BR100,New User,House,4,0.7515,228.66,4.82,49.97,false
2026-09-02T17:00:00.000Z,BR03,Existing User,Apartment,3,0.0656,248.05,1.31,49.97,false
2026-09-02T17:00:00.000Z,BR100,New User,House,4,0.6752,229.5,4.47,50.0,false
2026-09-02T16:45:00.000Z,BR100,New User,House,4,0.7412,230.13,4.43,49.99,false
2026-09-02T16:45:00.000Z,BR03,Existing User,Apartment,3,0.073,250.47,1.43,50.02,false
2026-09-02T16:30:00.000Z,BR03,Existing User,Apartment,3,0.0681,249.66,1.34,50.0,false
2026-09-02T16:30:00.000Z,BR100,New User,House,4,0.7182,229.39,4.67,50.06,false
2026-09-02T16:15:00.000Z,BR100,New User,House,4,0.7596,232.4,4.81,50.0,false


In [0]:
user1 = register_user(
    meter_id="BR03",
    monthly_target_kwh=200,
    monthly_budget=1500,
    home_type="Apartment",
    occupants=3
)

print(user1)

user2 = register_user(
    meter_id="BR100",
    monthly_target_kwh=250,
    monthly_budget=1800,
    home_type="House",
    occupants=4
)

print(user2)

✅ Existing user found!
Meter ID: BR03
Historical profile will be used.
{'meter_id': 'BR03', 'user_type': 'Existing User', 'home_type': 'Apartment', 'occupants': 3, 'monthly_target_kwh': 200.0, 'monthly_budget': 1500.0, 'baseline_energy_3min': 0.013, 'baseline_voltage': 247.12, 'baseline_current': 1.3, 'baseline_frequency': 50.01}
🆕 New user detected!
Meter ID: BR100
Synthetic smart-meter profile created.
{'meter_id': 'BR100', 'user_type': 'New User', 'home_type': 'House', 'occupants': 4, 'monthly_target_kwh': 250.0, 'monthly_budget': 1800.0, 'baseline_energy_3min': 0.15, 'baseline_voltage': 230.0, 'baseline_current': 4.5, 'baseline_frequency': 50.0}


In [0]:
# All users who should receive live smart-meter readings

registered_users = [
    user1,   # Existing user: BR03
    user2    # New user: BR100
]

print("Registered users:")
for user in registered_users:
    print(
        f"Meter: {user['meter_id']} | "
        f"Type: {user['user_type']} | "
        f"Home: {user['home_type']} | "
        f"Occupants: {user['occupants']}"
    )

Registered users:
Meter: BR03 | Type: Existing User | Home: Apartment | Occupants: 3
Meter: BR100 | Type: New User | Home: House | Occupants: 4


In [0]:
import time
from datetime import datetime, timedelta

INTERVAL_SECONDS = 2

# Every 2 real seconds = 15 simulated minutes
SIMULATED_MINUTES_PER_CYCLE = 15

# Start simulation from the 1st day of the current month
simulated_time = datetime.now().replace(
    day=1,
    hour=0,
    minute=0,
    second=0,
    microsecond=0
)

print("🔴 LIVE SMART-ENERGY STREAM STARTED")
print("Real update interval: 2 seconds")
print("Each update represents: 15 simulated minutes")
print("Simulation starts from the 1st day of the month")
print("Press STOP/INTERRUPT to stop.")
print("=" * 70)

while True:

    cycle_start = datetime.now()

    print(f"\n⏱️ Real update: {cycle_start}")
    print(f"🕒 Simulated time: {simulated_time}")
    print("-" * 70)

    for user in registered_users:

        try:

            reading = generate_reading(user)

            # Replace real timestamp with simulated timestamp
            reading["timestamp"] = simulated_time

            reading["user_type"] = user["user_type"]
            reading["home_type"] = user["home_type"]
            reading["occupants"] = int(user["occupants"])

            live_reading_df = spark.createDataFrame([reading])

            (
                live_reading_df
                .write
                .format("delta")
                .mode("append")
                .saveAsTable(live_table_name)
            )

            print(
                f"✅ {reading['meter_id']} | "
                f"Time: {reading['timestamp']} | "
                f"Energy: {reading['energy_kwh']} kWh | "
                f"Voltage: {reading['avg_voltage']} V | "
                f"Current: {reading['avg_current']} A | "
                f"Frequency: {reading['frequency']} Hz | "
                f"Abnormal: {reading['abnormal_event']}"
            )

        except Exception as e:

            print(
                f"❌ Error for {user['meter_id']}: {str(e)}"
            )

    # Move simulated time forward by 15 minutes
    simulated_time += timedelta(
        minutes=SIMULATED_MINUTES_PER_CYCLE
    )

    print("-" * 70)
    print("💤 Waiting 2 real seconds...")

    time.sleep(INTERVAL_SECONDS)

🔴 LIVE SMART-ENERGY STREAM STARTED
Real update interval: 2 seconds
Each update represents: 15 simulated minutes
Simulation starts from the 1st day of the month
Press STOP/INTERRUPT to stop.

⏱️ Real update: 2026-09-20 07:10:37.513824
🕒 Simulated time: 2026-09-01 00:00:00
----------------------------------------------------------------------
✅ BR03 | Time: 2026-09-01 00:00:00 | Energy: 0.0496 kWh | Voltage: 246.99 V | Current: 1.04 A | Frequency: 50.06 Hz | Abnormal: False
✅ BR100 | Time: 2026-09-01 00:00:00 | Energy: 0.7491 kWh | Voltage: 229.8 V | Current: 4.56 A | Frequency: 50.0 Hz | Abnormal: False
----------------------------------------------------------------------
💤 Waiting 2 real seconds...

⏱️ Real update: 2026-09-20 07:10:42.069146
🕒 Simulated time: 2026-09-01 00:15:00
----------------------------------------------------------------------
✅ BR03 | Time: 2026-09-01 00:15:00 | Energy: 0.0704 kWh | Voltage: 249.77 V | Current: 1.34 A | Frequency: 49.99 Hz | Abnormal: False
✅ BR

In [0]:
latest_live_df = spark.sql(f"""
    SELECT *
    FROM {live_table_name}
    ORDER BY timestamp DESC
    LIMIT 20
""")

display(latest_live_df)

timestamp,meter_id,user_type,home_type,occupants,energy_kwh,avg_voltage,avg_current,frequency,abnormal_event
2026-09-20T07:10:31.559Z,BR03,Existing User,Apartment,3,0.0652,243.77,1.4,50.0,false
2026-09-02T17:15:00.000Z,BR100,New User,House,4,0.7515,228.66,4.82,49.97,false
2026-09-02T17:15:00.000Z,BR03,Existing User,Apartment,3,0.0766,248.86,1.47,50.02,false
2026-09-02T17:00:00.000Z,BR100,New User,House,4,0.6752,229.5,4.47,50.0,false
2026-09-02T17:00:00.000Z,BR03,Existing User,Apartment,3,0.0656,248.05,1.31,49.97,false
2026-09-02T16:45:00.000Z,BR100,New User,House,4,0.7412,230.13,4.43,49.99,false
2026-09-02T16:45:00.000Z,BR03,Existing User,Apartment,3,0.073,250.47,1.43,50.02,false
2026-09-02T16:30:00.000Z,BR100,New User,House,4,0.7182,229.39,4.67,50.06,false
2026-09-02T16:30:00.000Z,BR03,Existing User,Apartment,3,0.0681,249.66,1.34,50.0,false
2026-09-02T16:15:00.000Z,BR100,New User,House,4,0.7596,232.4,4.81,50.0,false


In [0]:
from sklearn.ensemble import IsolationForest

# Load historical data for anomaly model
historical_anomaly_df = historical_df.select(
    "energy_kwh",
    "avg_voltage",
    "avg_current",
    "frequency"
)

historical_anomaly_pd = historical_anomaly_df.toPandas()

# Train Isolation Forest
live_iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)

live_iso_model.fit(
    historical_anomaly_pd[
        [
            "energy_kwh",
            "avg_voltage",
            "avg_current",
            "frequency"
        ]
    ]
)

print("✅ Live Isolation Forest model trained!")

✅ Live Isolation Forest model trained!


In [0]:
# Get all live readings
live_anomaly_df = spark.table(live_table_name)

# Convert required columns to Pandas
live_anomaly_pd = live_anomaly_df.select(
    "timestamp",
    "meter_id",
    "energy_kwh",
    "avg_voltage",
    "avg_current",
    "frequency"
).toPandas()

# Features used by Isolation Forest
anomaly_features = [
    "energy_kwh",
    "avg_voltage",
    "avg_current",
    "frequency"
]

# Predict anomalies
live_anomaly_pd["anomaly_prediction"] = live_iso_model.predict(
    live_anomaly_pd[anomaly_features]
)

# Convert prediction into readable status
live_anomaly_pd["anomaly_status"] = live_anomaly_pd[
    "anomaly_prediction"
].apply(
    lambda x: "ANOMALY DETECTED" if x == -1 else "NORMAL"
)

print("✅ Isolation Forest predictions completed!")

display(live_anomaly_pd.tail(20))

✅ Isolation Forest predictions completed!


timestamp,meter_id,energy_kwh,avg_voltage,avg_current,frequency,anomaly_prediction,anomaly_status
2026-09-01T19:15:00.000Z,BR03,0.0701,246.33,1.31,50.01,1,NORMAL
2026-09-01T20:15:00.000Z,BR03,0.0725,250.03,1.52,50.05,1,NORMAL
2026-09-01T20:45:00.000Z,BR03,0.0688,244.73,1.26,50.01,1,NORMAL
2026-09-01T20:00:00.000Z,BR03,0.0665,247.96,1.35,49.97,1,NORMAL
2026-09-01T18:15:00.000Z,BR03,0.0556,246.51,1.04,50.04,1,NORMAL
2026-09-01T20:30:00.000Z,BR03,0.062,248.32,1.19,50.09,1,NORMAL
2026-09-01T18:45:00.000Z,BR03,0.063,249.65,1.39,50.04,1,NORMAL
2026-09-01T19:30:00.000Z,BR03,0.06,250.34,1.27,49.97,1,NORMAL
2026-09-01T19:00:00.000Z,BR03,0.062,248.7,1.21,50.05,1,NORMAL
2026-09-01T19:30:00.000Z,BR100,0.6315,233.42,3.93,49.95,1,NORMAL


In [0]:
live_table_name = "workspace.default.smart_energy_live"

print("✅ Live table:", live_table_name)

✅ Live table: workspace.default.smart_energy_live


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

live_raw_df = spark.table(live_table_name)

latest_window = (
    Window
    .partitionBy("meter_id")
    .orderBy(F.col("timestamp").desc())
)

latest_readings = (
    live_raw_df
    .withColumn(
        "rank",
        F.row_number().over(latest_window)
    )
    .filter(F.col("rank") == 1)
    .select(
        "meter_id",
        F.col("timestamp").alias("last_update"),
        F.col("energy_kwh").alias("latest_energy_kwh"),
        F.col("avg_voltage").alias("latest_voltage"),
        F.col("avg_current").alias("latest_current"),
        F.col("frequency").alias("latest_frequency"),
        "user_type",
        "home_type",
        "occupants"
    )
)

display(latest_readings)

meter_id,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,user_type,home_type,occupants
BR03,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,Existing User,Apartment,3
BR100,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,New User,House,4


In [0]:
simulation_date = (
    live_raw_df
    .select(
        F.to_date(F.max("timestamp")).alias("simulation_date")
    )
    .collect()[0]["simulation_date"]
)

print("Simulation date:", simulation_date)

daily_usage = (
    live_raw_df
    .filter(
        F.to_date("timestamp") == F.lit(simulation_date)
    )
    .groupBy("meter_id")
    .agg(
        F.sum("energy_kwh").alias("today_energy_kwh")
    )
)

display(daily_usage)

Simulation date: 2026-09-20


meter_id,today_energy_kwh
BR03,0.0652


In [0]:
anomaly_spark_df = spark.createDataFrame(
    live_anomaly_pd[
        [
            "meter_id",
            "timestamp",
            "anomaly_prediction"
        ]
    ]
)

anomaly_counts = (
    anomaly_spark_df
    .filter(
        F.col("anomaly_prediction") == -1
    )
    .groupBy("meter_id")
    .agg(
        F.count("*").alias("anomaly_count")
    )
)

display(anomaly_counts)

meter_id,anomaly_count
BR100,11


In [0]:
live_metrics_df = (
    latest_readings
    .join(
        daily_usage,
        on="meter_id",
        how="left"
    )
    .join(
        anomaly_counts,
        on="meter_id",
        how="left"
    )
    .fillna({
        "today_energy_kwh": 0.0,
        "anomaly_count": 0
    })
)

display(live_metrics_df)

meter_id,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,user_type,home_type,occupants,today_energy_kwh,anomaly_count
BR03,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,Existing User,Apartment,3,0.0652,0
BR100,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,New User,House,4,0.0,11


In [0]:
user_settings_df = spark.createDataFrame(
    [
        (
            user1["meter_id"],
            float(user1["monthly_target_kwh"]),
            float(user1["monthly_budget"])
        ),
        (
            user2["meter_id"],
            float(user2["monthly_target_kwh"]),
            float(user2["monthly_budget"])
        )
    ],
    [
        "meter_id",
        "monthly_target_kwh",
        "monthly_budget"
    ]
)

live_metrics_df = (
    live_metrics_df
    .join(
        user_settings_df,
        on="meter_id",
        how="left"
    )
)

display(live_metrics_df)

meter_id,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,user_type,home_type,occupants,today_energy_kwh,anomaly_count,monthly_target_kwh,monthly_budget
BR03,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,Existing User,Apartment,3,0.0652,0,200.0,1500.0
BR100,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,New User,House,4,0.0,11,250.0,1800.0


In [0]:
live_metrics_df = (
    live_metrics_df
    .withColumn(
        "current_day",
        F.dayofmonth(F.lit(simulation_date))
    )
    .withColumn(
        "projected_monthly_usage",
        F.when(
            F.col("current_day") > 0,
            (
                F.col("today_energy_kwh")
                / F.col("current_day")
            ) * 30
        ).otherwise(0)
    )
    .withColumn(
        "projected_usage_difference",
        F.col("projected_monthly_usage")
        - F.col("monthly_target_kwh")
    )
    .withColumn(
        "guardian_status",
        F.when(
            F.col("projected_monthly_usage")
            > F.col("monthly_target_kwh"),
            "USAGE TARGET RISK"
        )
        .otherwise("ON TRACK")
    )
)

display(live_metrics_df)

meter_id,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,user_type,home_type,occupants,today_energy_kwh,anomaly_count,monthly_target_kwh,monthly_budget,current_day,projected_monthly_usage,projected_usage_difference,guardian_status
BR03,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,Existing User,Apartment,3,0.0652,0,200.0,1500.0,20,0.0978,-199.9022,ON TRACK
BR100,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,New User,House,4,0.0,11,250.0,1800.0,20,0.0,-250.0,ON TRACK


In [0]:
live_metrics_df = (
    live_metrics_df

    .withColumn(
        "projected_bill",

        F.when(
            F.col("projected_monthly_usage") <= 50,

            F.col("projected_monthly_usage") * 3
        )

        .when(
            F.col("projected_monthly_usage") <= 150,

            (50 * 3)
            +
            (
                (F.col("projected_monthly_usage") - 50)
                * 5
            )
        )

        .when(
            F.col("projected_monthly_usage") <= 250,

            (50 * 3)
            +
            (100 * 5)
            +
            (
                (F.col("projected_monthly_usage") - 150)
                * 7
            )
        )

        .otherwise(

            (50 * 3)
            +
            (100 * 5)
            +
            (100 * 7)
            +
            (
                (F.col("projected_monthly_usage") - 250)
                * 10
            )
        )
    )
)

display(live_metrics_df)

meter_id,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,user_type,home_type,occupants,today_energy_kwh,anomaly_count,monthly_target_kwh,monthly_budget,current_day,projected_monthly_usage,projected_usage_difference,guardian_status,projected_bill
BR03,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,Existing User,Apartment,3,0.0652,0,200.0,1500.0,20,0.0978,-199.9022,ON TRACK,0.2934
BR100,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,New User,House,4,0.0,11,250.0,1800.0,20,0.0,-250.0,ON TRACK,0.0


In [0]:
live_metrics_df = (
    live_metrics_df

    .withColumn(
        "budget_difference",
        F.col("monthly_budget")
        - F.col("projected_bill")
    )

    .withColumn(
        "budget_status",
        F.when(
            F.col("projected_bill")
            > F.col("monthly_budget"),
            "BUDGET RISK"
        )
        .otherwise("WITHIN BUDGET")
    )

    .withColumn(
        "anomaly_status",
        F.when(
            F.col("anomaly_count") > 0,
            "ANOMALY DETECTED"
        )
        .otherwise("NORMAL")
    )
)

display(live_metrics_df)

meter_id,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,user_type,home_type,occupants,today_energy_kwh,anomaly_count,monthly_target_kwh,monthly_budget,current_day,projected_monthly_usage,projected_usage_difference,guardian_status,projected_bill,budget_difference,budget_status,anomaly_status
BR03,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,Existing User,Apartment,3,0.0652,0,200.0,1500.0,20,0.0978,-199.9022,ON TRACK,0.2934,1499.7066,WITHIN BUDGET,NORMAL
BR100,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,New User,House,4,0.0,11,250.0,1800.0,20,0.0,-250.0,ON TRACK,0.0,1800.0,WITHIN BUDGET,ANOMALY DETECTED


In [0]:
dashboard_live_df = live_metrics_df.select(
    "meter_id",
    "user_type",
    "home_type",
    "occupants",

    "last_update",

    # Live readings
    "latest_energy_kwh",
    "latest_voltage",
    "latest_current",
    "latest_frequency",

    # Consumption
    "today_energy_kwh",

    # User planning
    "monthly_target_kwh",
    "monthly_budget",

    # Budget Guardian
    "projected_monthly_usage",
    "projected_bill",
    "projected_usage_difference",
    "budget_difference",

    # Status
    "guardian_status",
    "budget_status",

    # Anomaly
    "anomaly_count",
    "anomaly_status"
)

display(dashboard_live_df)

meter_id,user_type,home_type,occupants,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,today_energy_kwh,monthly_target_kwh,monthly_budget,projected_monthly_usage,projected_bill,projected_usage_difference,budget_difference,guardian_status,budget_status,anomaly_count,anomaly_status
BR03,Existing User,Apartment,3,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,0.0652,200.0,1500.0,0.0978,0.2934,-199.9022,1499.7066,ON TRACK,WITHIN BUDGET,0,NORMAL
BR100,New User,House,4,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,0.0,250.0,1800.0,0.0,0.0,-250.0,1800.0,ON TRACK,WITHIN BUDGET,11,ANOMALY DETECTED


In [0]:
dashboard_table_name = "workspace.default.smart_energy_dashboard"

(
    dashboard_live_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(dashboard_table_name)
)

print("✅ Dashboard table updated successfully!")

✅ Dashboard table updated successfully!


In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {dashboard_table_name}
        ORDER BY last_update DESC
    """)
)

meter_id,user_type,home_type,occupants,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,today_energy_kwh,monthly_target_kwh,monthly_budget,projected_monthly_usage,projected_bill,projected_usage_difference,budget_difference,guardian_status,budget_status,anomaly_count,anomaly_status
BR03,Existing User,Apartment,3,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,0.0652,200.0,1500.0,0.0978,0.2934,-199.9022,1499.7066,ON TRACK,WITHIN BUDGET,0,NORMAL
BR100,New User,House,4,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,0.0,250.0,1800.0,0.0,0.0,-250.0,1800.0,ON TRACK,WITHIN BUDGET,11,ANOMALY DETECTED


In [0]:
%pip install xgboost

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F

# Reload the original historical CSV
file_path = "/Volumes/workspace/default/data_s/SM Cleaned Data BR2019.csv"

historical_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

# Rename original columns
historical_forecast_df = (
    historical_raw_df
    .withColumnRenamed("x_Timestamp", "timestamp")
    .withColumnRenamed("t_kWh", "energy_kwh")
    .withColumnRenamed("z_Avg Voltage (Volt)", "avg_voltage")
    .withColumnRenamed("z_Avg Current (Amp)", "avg_current")
    .withColumnRenamed("y_Freq (Hz)", "frequency")
    .withColumnRenamed("meter", "meter_id")
)

# Make sure timestamp is actually TimestampType
historical_forecast_df = (
    historical_forecast_df
    .withColumn(
        "timestamp",
        F.to_timestamp("timestamp")
    )
)

print("✅ Historical forecasting data loaded!")
print("Columns:")
print(historical_forecast_df.columns)

# Convert 3-minute readings into hourly readings
historical_hourly_df = (
    historical_forecast_df
    .withColumn(
        "hour_timestamp",
        F.date_trunc("hour", "timestamp")
    )
    .groupBy(
        "meter_id",
        "hour_timestamp"
    )
    .agg(
        F.sum("energy_kwh").alias("energy_kwh"),
        F.avg("avg_voltage").alias("avg_voltage"),
        F.avg("avg_current").alias("avg_current"),
        F.avg("frequency").alias("frequency")
    )
    .orderBy(
        "meter_id",
        "hour_timestamp"
    )
)

print("✅ Hourly historical data created!")
display(historical_hourly_df.limit(10))

✅ Historical forecasting data loaded!
Columns:
['timestamp', 'energy_kwh', 'avg_voltage', 'avg_current', 'frequency', 'meter_id']
✅ Hourly historical data created!


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency
BR02,2019-07-10T00:00:00.000Z,0.4050000000000001,242.099,1.7680000000000002,50.023999999999994
BR02,2019-07-10T01:00:00.000Z,0.4460000000000001,244.24099999999999,1.905,50.04699999999999
BR02,2019-07-10T02:00:00.000Z,0.4470000000000001,246.53199999999998,1.8895000000000004,50.0355
BR02,2019-07-10T03:00:00.000Z,0.4500000000000001,248.40900000000002,1.8814999999999997,50.05850000000002
BR02,2019-07-10T04:00:00.000Z,0.4540000000000001,249.70299999999997,1.8869999999999998,50.037
BR02,2019-07-10T05:00:00.000Z,0.4540000000000001,249.80050000000006,1.8830000000000002,49.974999999999994
BR02,2019-07-10T06:00:00.000Z,0.28800000000000014,251.82549999999998,1.171,49.98199999999999
BR02,2019-07-10T07:00:00.000Z,0.26300000000000007,255.01200000000003,1.0664999999999998,50.07349999999999
BR02,2019-07-10T08:00:00.000Z,0.2390000000000001,252.54650000000007,1.0299999999999998,50.1075
BR02,2019-07-10T09:00:00.000Z,0.29800000000000004,236.49099999999993,1.5410000000000001,47.55449999999998


In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Time-based features
forecast_df = (
    historical_hourly_df

    .withColumn(
        "hour",
        F.hour("hour_timestamp")
    )

    .withColumn(
        "day",
        F.dayofmonth("hour_timestamp")
    )

    .withColumn(
        "month",
        F.month("hour_timestamp")
    )

    .withColumn(
        "year",
        F.year("hour_timestamp")
    )

    .withColumn(
        "day_of_week",
        F.dayofweek("hour_timestamp")
    )

    .withColumn(
        "is_weekend",
        F.when(
            F.dayofweek("hour_timestamp").isin([1, 7]),
            1
        ).otherwise(0)
    )
)

print("✅ Time features created!")

display(forecast_df.limit(10))

✅ Time features created!


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,year,day_of_week,is_weekend
BR02,2019-07-10T00:00:00.000Z,0.4050000000000001,242.099,1.7680000000000002,50.023999999999994,0,10,7,2019,4,0
BR02,2019-07-10T01:00:00.000Z,0.4460000000000001,244.24099999999999,1.905,50.04699999999999,1,10,7,2019,4,0
BR02,2019-07-10T02:00:00.000Z,0.4470000000000001,246.53199999999998,1.8895000000000004,50.0355,2,10,7,2019,4,0
BR02,2019-07-10T03:00:00.000Z,0.4500000000000001,248.40900000000002,1.8814999999999997,50.05850000000002,3,10,7,2019,4,0
BR02,2019-07-10T04:00:00.000Z,0.4540000000000001,249.70299999999997,1.8869999999999998,50.037,4,10,7,2019,4,0
BR02,2019-07-10T05:00:00.000Z,0.4540000000000001,249.80050000000006,1.8830000000000002,49.974999999999994,5,10,7,2019,4,0
BR02,2019-07-10T06:00:00.000Z,0.28800000000000014,251.82549999999998,1.171,49.98199999999999,6,10,7,2019,4,0
BR02,2019-07-10T07:00:00.000Z,0.26300000000000007,255.01200000000003,1.0664999999999998,50.07349999999999,7,10,7,2019,4,0
BR02,2019-07-10T08:00:00.000Z,0.2390000000000001,252.54650000000007,1.0299999999999998,50.1075,8,10,7,2019,4,0
BR02,2019-07-10T09:00:00.000Z,0.29800000000000004,236.49099999999993,1.5410000000000001,47.55449999999998,9,10,7,2019,4,0


In [0]:
forecast_window = (
    Window
    .partitionBy("meter_id")
    .orderBy("hour_timestamp")
)

forecast_df = (
    forecast_df

    .withColumn(
        "energy_lag_1",
        F.lag("energy_kwh", 1).over(forecast_window)
    )

    .withColumn(
        "energy_lag_2",
        F.lag("energy_kwh", 2).over(forecast_window)
    )

    .withColumn(
        "energy_lag_24",
        F.lag("energy_kwh", 24).over(forecast_window)
    )

    .withColumn(
        "energy_lag_168",
        F.lag("energy_kwh", 168).over(forecast_window)
    )
)

print("✅ Lag features created!")

display(forecast_df.limit(10))

✅ Lag features created!


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,year,day_of_week,is_weekend,energy_lag_1,energy_lag_2,energy_lag_24,energy_lag_168
BR06,2019-07-11T00:00:00.000Z,0.114,77.40950000000001,0.7100000000000001,15.009500000000003,0,11,7,2019,5,0,null,null,null,null
BR06,2019-07-11T01:00:00.000Z,0.021,26.737000000000002,0.1935,5.006,1,11,7,2019,5,0,0.114,null,null,null
BR06,2019-07-11T02:00:00.000Z,0.09999999999999999,65.1115,0.6405000000000001,12.514,2,11,7,2019,5,0,0.021,0.114,null,null
BR06,2019-07-11T03:00:00.000Z,0.10300000000000001,80.78450000000001,0.639,15.016499999999999,3,11,7,2019,5,0,0.09999999999999999,0.021,null,null
BR06,2019-07-11T04:00:00.000Z,0.318,201.3635,2.0029999999999997,37.549499999999995,4,11,7,2019,5,0,0.10300000000000001,0.09999999999999999,null,null
BR06,2019-07-11T05:00:00.000Z,0.129,67.22399999999999,0.8230000000000001,12.5025,5,11,7,2019,5,0,0.318,0.10300000000000001,null,null
BR06,2019-07-11T06:00:00.000Z,0.8890000000000001,267.87149999999997,4.086999999999999,49.981,6,11,7,2019,5,0,0.129,0.318,null,null
BR06,2019-07-11T07:00:00.000Z,0.6110000000000003,268.9845,3.339,50.039500000000004,7,11,7,2019,5,0,0.8890000000000001,0.129,null,null
BR06,2019-07-11T08:00:00.000Z,0.5240000000000002,240.1715,2.4605,45.0735,8,11,7,2019,5,0,0.6110000000000003,0.8890000000000001,null,null
BR06,2019-07-11T09:00:00.000Z,0.5800000000000002,261.06199999999995,2.9004999999999996,50.05799999999999,9,11,7,2019,5,0,0.5240000000000002,0.6110000000000003,null,null


In [0]:
forecast_window = (
    Window
    .partitionBy("meter_id")
    .orderBy("hour_timestamp")
)

forecast_df = (
    forecast_df

    .withColumn(
        "energy_lag_1",
        F.lag("energy_kwh", 1).over(forecast_window)
    )

    .withColumn(
        "energy_lag_2",
        F.lag("energy_kwh", 2).over(forecast_window)
    )

    .withColumn(
        "energy_lag_24",
        F.lag("energy_kwh", 24).over(forecast_window)
    )

    .withColumn(
        "energy_lag_168",
        F.lag("energy_kwh", 168).over(forecast_window)
    )
)

print("✅ Lag features created!")

display(forecast_df.limit(10))

✅ Lag features created!


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,year,day_of_week,is_weekend,energy_lag_1,energy_lag_2,energy_lag_24,energy_lag_168
BR06,2019-07-11T00:00:00.000Z,0.114,77.40950000000001,0.7100000000000001,15.009500000000003,0,11,7,2019,5,0,null,null,null,null
BR06,2019-07-11T01:00:00.000Z,0.021,26.737000000000002,0.1935,5.006,1,11,7,2019,5,0,0.114,null,null,null
BR06,2019-07-11T02:00:00.000Z,0.09999999999999999,65.1115,0.6405000000000001,12.514,2,11,7,2019,5,0,0.021,0.114,null,null
BR06,2019-07-11T03:00:00.000Z,0.10300000000000001,80.78450000000001,0.639,15.016499999999999,3,11,7,2019,5,0,0.09999999999999999,0.021,null,null
BR06,2019-07-11T04:00:00.000Z,0.318,201.3635,2.0029999999999997,37.549499999999995,4,11,7,2019,5,0,0.10300000000000001,0.09999999999999999,null,null
BR06,2019-07-11T05:00:00.000Z,0.129,67.22399999999999,0.8230000000000001,12.5025,5,11,7,2019,5,0,0.318,0.10300000000000001,null,null
BR06,2019-07-11T06:00:00.000Z,0.8890000000000001,267.87149999999997,4.086999999999999,49.981,6,11,7,2019,5,0,0.129,0.318,null,null
BR06,2019-07-11T07:00:00.000Z,0.6110000000000003,268.9845,3.339,50.039500000000004,7,11,7,2019,5,0,0.8890000000000001,0.129,null,null
BR06,2019-07-11T08:00:00.000Z,0.5240000000000002,240.1715,2.4605,45.0735,8,11,7,2019,5,0,0.6110000000000003,0.8890000000000001,null,null
BR06,2019-07-11T09:00:00.000Z,0.5800000000000002,261.06199999999995,2.9004999999999996,50.05799999999999,9,11,7,2019,5,0,0.5240000000000002,0.6110000000000003,null,null


In [0]:
    rolling_window = (
    Window
    .partitionBy("meter_id")
    .orderBy("hour_timestamp")
    .rowsBetween(-24, -1)
)

forecast_df = (
    forecast_df
    .withColumn(
        "rolling_avg_24h",
        F.avg("energy_kwh").over(rolling_window)
    )
)

print("✅ 24-hour rolling average created!")

display(forecast_df.limit(10))

✅ 24-hour rolling average created!


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,year,day_of_week,is_weekend,energy_lag_1,energy_lag_2,energy_lag_24,energy_lag_168,rolling_avg_24h
BR06,2019-07-11T00:00:00.000Z,0.114,77.40950000000001,0.7100000000000001,15.009500000000003,0,11,7,2019,5,0,null,null,null,null,null
BR06,2019-07-11T01:00:00.000Z,0.021,26.737000000000002,0.1935,5.006,1,11,7,2019,5,0,0.114,null,null,null,0.114
BR06,2019-07-11T02:00:00.000Z,0.09999999999999999,65.1115,0.6405000000000001,12.514,2,11,7,2019,5,0,0.021,0.114,null,null,0.0675
BR06,2019-07-11T03:00:00.000Z,0.10300000000000001,80.78450000000001,0.639,15.016499999999999,3,11,7,2019,5,0,0.09999999999999999,0.021,null,null,0.07833333333333332
BR06,2019-07-11T04:00:00.000Z,0.318,201.3635,2.0029999999999997,37.549499999999995,4,11,7,2019,5,0,0.10300000000000001,0.09999999999999999,null,null,0.08449999999999999
BR06,2019-07-11T05:00:00.000Z,0.129,67.22399999999999,0.8230000000000001,12.5025,5,11,7,2019,5,0,0.318,0.10300000000000001,null,null,0.13119999999999998
BR06,2019-07-11T06:00:00.000Z,0.8890000000000001,267.87149999999997,4.086999999999999,49.981,6,11,7,2019,5,0,0.129,0.318,null,null,0.13083333333333333
BR06,2019-07-11T07:00:00.000Z,0.6110000000000003,268.9845,3.339,50.039500000000004,7,11,7,2019,5,0,0.8890000000000001,0.129,null,null,0.23914285714285713
BR06,2019-07-11T08:00:00.000Z,0.5240000000000002,240.1715,2.4605,45.0735,8,11,7,2019,5,0,0.6110000000000003,0.8890000000000001,null,null,0.285625
BR06,2019-07-11T09:00:00.000Z,0.5800000000000002,261.06199999999995,2.9004999999999996,50.05799999999999,9,11,7,2019,5,0,0.5240000000000002,0.6110000000000003,null,null,0.3121111111111111


In [0]:
forecast_df = (
    forecast_df
    .withColumn(
        "target_energy",
        F.lead("energy_kwh", 1).over(forecast_window)
    )
)

print("✅ Next-hour forecasting target created!")

display(
    forecast_df.select(
        "meter_id",
        "hour_timestamp",
        "energy_kwh",
        "energy_lag_1",
        "energy_lag_24",
        "target_energy"
    ).limit(10)
)

✅ Next-hour forecasting target created!


meter_id,hour_timestamp,energy_kwh,energy_lag_1,energy_lag_24,target_energy
BR02,2019-07-10T00:00:00.000Z,0.4050000000000001,null,null,0.4460000000000001
BR02,2019-07-10T01:00:00.000Z,0.4460000000000001,0.4050000000000001,null,0.4470000000000001
BR02,2019-07-10T02:00:00.000Z,0.4470000000000001,0.4460000000000001,null,0.4500000000000001
BR02,2019-07-10T03:00:00.000Z,0.4500000000000001,0.4470000000000001,null,0.4540000000000001
BR02,2019-07-10T04:00:00.000Z,0.4540000000000001,0.4500000000000001,null,0.4540000000000001
BR02,2019-07-10T05:00:00.000Z,0.4540000000000001,0.4540000000000001,null,0.28800000000000014
BR02,2019-07-10T06:00:00.000Z,0.28800000000000014,0.4540000000000001,null,0.26300000000000007
BR02,2019-07-10T07:00:00.000Z,0.26300000000000007,0.28800000000000014,null,0.2390000000000001
BR02,2019-07-10T08:00:00.000Z,0.2390000000000001,0.26300000000000007,null,0.29800000000000004
BR02,2019-07-10T09:00:00.000Z,0.29800000000000004,0.2390000000000001,null,0.17800000000000002


In [0]:
model_forecast_df = forecast_df.dropna(
    subset=[
        "energy_lag_1",
        "energy_lag_2",
        "energy_lag_24",
        "energy_lag_168",
        "rolling_avg_24h",
        "target_energy"
    ]
)

print("✅ Complete forecasting dataset created!")
print("Rows:", model_forecast_df.count())

display(model_forecast_df.limit(10))

✅ Complete forecasting dataset created!
Rows: 138198


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,year,day_of_week,is_weekend,energy_lag_1,energy_lag_2,energy_lag_24,energy_lag_168,rolling_avg_24h,target_energy
BR06,2019-07-30T00:00:00.000Z,0.8400000000000001,237.38950000000006,3.6270000000000002,50.02599999999999,0,30,7,2019,3,0,0.3760000000000001,0.724,0.28400000000000014,0.114,0.539375,0.8380000000000002
BR06,2019-07-30T01:00:00.000Z,0.8380000000000002,235.35549999999998,3.691499999999999,50.017999999999994,1,30,7,2019,3,0,0.8400000000000001,0.3760000000000001,0.27400000000000013,0.021,0.5625416666666668,0.8220000000000002
BR06,2019-07-30T02:00:00.000Z,0.8220000000000002,237.7935,3.5909999999999997,50.003,2,30,7,2019,3,0,0.8380000000000002,0.8400000000000001,0.21000000000000008,0.09999999999999999,0.5860416666666668,0.7780000000000001
BR06,2019-07-30T03:00:00.000Z,0.7780000000000001,241.28249999999997,3.3625000000000007,49.9705,3,30,7,2019,3,0,0.8220000000000002,0.8380000000000002,0.2060000000000001,0.10300000000000001,0.6115416666666669,0.8250000000000001
BR06,2019-07-30T04:00:00.000Z,0.8250000000000001,242.961,3.5985,49.975999999999985,4,30,7,2019,3,0,0.7780000000000001,0.8220000000000002,0.2080000000000001,0.318,0.6353750000000001,0.6300000000000002
BR06,2019-07-30T05:00:00.000Z,0.6300000000000002,248.38899999999998,3.1394999999999995,49.93599999999999,5,30,7,2019,3,0,0.8250000000000001,0.7780000000000001,0.2510000000000001,0.129,0.6610833333333335,0.9510000000000001
BR06,2019-07-30T06:00:00.000Z,0.9510000000000001,254.448,4.311999999999999,49.8975,6,30,7,2019,3,0,0.6300000000000002,0.8250000000000001,0.49500000000000016,0.8890000000000001,0.6768750000000002,0.28200000000000003
BR06,2019-07-30T07:00:00.000Z,0.28200000000000003,254.9,1.2655000000000005,49.99399999999999,7,30,7,2019,3,0,0.9510000000000001,0.6300000000000002,0.5600000000000002,0.6110000000000003,0.6958750000000001,0.43300000000000016
BR06,2019-07-30T08:00:00.000Z,0.43300000000000016,252.721,2.0780000000000003,50.030499999999996,8,30,7,2019,3,0,0.28200000000000003,0.9510000000000001,0.46600000000000014,0.5240000000000002,0.6842916666666667,1.035
BR06,2019-07-30T09:00:00.000Z,1.035,246.33450000000002,4.3685,49.9985,9,30,7,2019,3,0,0.43300000000000016,0.28200000000000003,1.578,0.5800000000000002,0.6829166666666668,0.5680000000000003


In [0]:
feature_cols = [
    "hour",
    "day",
    "month",
    "day_of_week",
    "is_weekend",
    "avg_voltage",
    "avg_current",
    "frequency",
    "energy_lag_1",
    "energy_lag_2",
    "energy_lag_24",
    "energy_lag_168",
    "rolling_avg_24h"
]

target_col = "target_energy"

print("Forecasting features:")
for feature in feature_cols:
    print("-", feature)

print("\nTarget:", target_col)

Forecasting features:
- hour
- day
- month
- day_of_week
- is_weekend
- avg_voltage
- avg_current
- frequency
- energy_lag_1
- energy_lag_2
- energy_lag_24
- energy_lag_168
- rolling_avg_24h

Target: target_energy


In [0]:
# Keep the data in chronological order
model_forecast_df = model_forecast_df.orderBy("hour_timestamp")

# Find the 80% time point
split_time = model_forecast_df.select(
    F.expr(
        "percentile_approx(hour_timestamp, 0.8)"
    ).alias("split_time")
).collect()[0]["split_time"]

print("Split time:", split_time)

# Training data = first 80%
train_forecast_df = model_forecast_df.filter(
    F.col("hour_timestamp") <= F.lit(split_time)
)

# Testing data = remaining 20%
test_forecast_df = model_forecast_df.filter(
    F.col("hour_timestamp") > F.lit(split_time)
)

print("Training rows:", train_forecast_df.count())
print("Testing rows :", test_forecast_df.count())

Split time: 2019-11-30 19:00:00
Training rows: 110552
Testing rows : 27646


In [0]:
train_pd = train_forecast_df.select(
    feature_cols + [target_col]
).toPandas()

test_pd = test_forecast_df.select(
    feature_cols + [target_col]
).toPandas()

# Training features and target
X_train = train_pd[feature_cols]
y_train = train_pd[target_col]

# Testing features and target
X_test = test_pd[feature_cols]
y_test = test_pd[target_col]

print("✅ Data prepared for ML!")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

✅ Data prepared for ML!
X_train: (110552, 13)
y_train: (110552,)
X_test : (27646, 13)
y_test : (27646,)


In [0]:
from xgboost import XGBRegressor

xgb_live_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_live_model.fit(
    X_train,
    y_train
)

print("✅ XGBoost forecasting model trained!")

✅ XGBoost forecasting model trained!


In [0]:
y_pred_xgb_live = xgb_live_model.predict(X_test)

print("✅ XGBoost predictions generated!")
print("Number of predictions:", len(y_pred_xgb_live))

✅ XGBoost predictions generated!
Number of predictions: 27646


In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

xgb_mae = mean_absolute_error(
    y_test,
    y_pred_xgb_live
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_xgb_live
    )
)

xgb_r2 = r2_score(
    y_test,
    y_pred_xgb_live
)

print("📊 XGBoost Forecasting Performance")
print("-----------------------------------")
print("MAE  :", round(xgb_mae, 4))
print("RMSE :", round(xgb_rmse, 4))
print("R²   :", round(xgb_r2, 4))

📊 XGBoost Forecasting Performance
-----------------------------------
MAE  : 0.1051
RMSE : 0.22
R²   : 0.5865


In [0]:
from sklearn.ensemble import RandomForestRegressor

rf_live_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_live_model.fit(
    X_train,
    y_train
)

print("✅ Random Forest forecasting model trained!")

✅ Random Forest forecasting model trained!


In [0]:
y_pred_rf_live = rf_live_model.predict(X_test)

print("✅ Random Forest predictions generated!")
print("Number of predictions:", len(y_pred_rf_live))

✅ Random Forest predictions generated!
Number of predictions: 27646


In [0]:
rf_mae = mean_absolute_error(
    y_test,
    y_pred_rf_live
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_rf_live
    )
)

rf_r2 = r2_score(
    y_test,
    y_pred_rf_live
)

print("📊 Random Forest Forecasting Performance")
print("-----------------------------------------")
print("MAE  :", round(rf_mae, 4))
print("RMSE :", round(rf_rmse, 4))
print("R²   :", round(rf_r2, 4))

📊 Random Forest Forecasting Performance
-----------------------------------------
MAE  : 0.1064
RMSE : 0.2192
R²   : 0.5896


In [0]:
comparison_df = spark.createDataFrame(
    [
        (
            "XGBoost",
            float(xgb_mae),
            float(xgb_rmse),
            float(xgb_r2)
        ),
        (
            "Random Forest",
            float(rf_mae),
            float(rf_rmse),
            float(rf_r2)
        )
    ],
    [
        "model",
        "MAE",
        "RMSE",
        "R2"
    ]
)

display(comparison_df)

model,MAE,RMSE,R2
XGBoost,0.10508349830171654,0.22004381963558758,0.5864663166194618
Random Forest,0.10635564433512025,0.21921599169096026,0.5895719779382245


In [0]:
# Read all live smart-meter data
live_stream_df = spark.table(live_table_name)

# Convert 2-minute readings into hourly consumption
live_hourly_df = (
    live_stream_df
    .withColumn(
        "hour_timestamp",
        F.date_trunc("hour", "timestamp")
    )
    .groupBy(
        "meter_id",
        "hour_timestamp"
    )
    .agg(
        F.sum("energy_kwh").alias("energy_kwh"),
        F.avg("avg_voltage").alias("avg_voltage"),
        F.avg("avg_current").alias("avg_current"),
        F.avg("frequency").alias("frequency")
    )
    .orderBy(
        "meter_id",
        "hour_timestamp"
    )
)

print("✅ Live data converted to hourly data!")

display(live_hourly_df)

✅ Live data converted to hourly data!


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency
BR03,2026-09-01T00:00:00.000Z,0.8829999999999999,241.56416666666667,1.5166666666666668,50.015833333333326
BR03,2026-09-01T01:00:00.000Z,0.8716999999999999,247.33166666666668,1.5225,50.010833333333345
BR03,2026-09-01T02:00:00.000Z,0.7929999999999999,242.65083333333334,1.3291666666666666,50.0025
BR03,2026-09-01T03:00:00.000Z,0.8422999999999999,246.82583333333335,1.388333333333333,50.020833333333336
BR03,2026-09-01T04:00:00.000Z,0.7839999999999998,247.04083333333332,1.2825,50.02
BR03,2026-09-01T05:00:00.000Z,0.7375,240.92166666666665,1.2275,50.01583333333334
BR03,2026-09-01T06:00:00.000Z,0.6285999999999999,248.323,1.295,50.017999999999994
BR03,2026-09-01T07:00:00.000Z,0.6151,247.93124999999998,1.59875,49.98625
BR03,2026-09-01T08:00:00.000Z,0.5265,246.69125,1.3350000000000002,50.01625
BR03,2026-09-01T09:00:00.000Z,0.5470999999999999,246.05374999999998,1.2925000000000002,49.9875


In [0]:
from datetime import datetime, timedelta
import random


registered_meter_ids = [
    user["meter_id"]
    for user in registered_users
]

historical_registered_df = (
    historical_hourly_df
    .filter(
        F.col("meter_id").isin(registered_meter_ids)
    )
)

# We need MORE than 168 rows because lag_168
# requires 168 previous observations.
history_window = (
    Window
    .partitionBy("meter_id")
    .orderBy(F.col("hour_timestamp").desc())
)

historical_seed_df = (
    historical_registered_df
    .withColumn(
        "row_num",
        F.row_number().over(history_window)
    )
    .filter(
        F.col("row_num") <= 200
    )
    .drop("row_num")
)

print("✅ Historical seed created!")
print("Using up to 200 hourly records per existing meter.")

display(
    historical_seed_df
    .orderBy("meter_id", "hour_timestamp")
    .limit(20)
)

✅ Historical seed created!
Using up to 200 hourly records per existing meter.


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency
BR03,2019-12-18T16:00:00.000Z,0.04600000000000001,243.68900000000002,0.20600000000000002,50.01050000000001
BR03,2019-12-18T17:00:00.000Z,0.04900000000000001,243.52549999999997,0.21750000000000008,50.020500000000006
BR03,2019-12-18T18:00:00.000Z,0.05500000000000001,243.24949999999998,0.27050000000000013,50.031499999999994
BR03,2019-12-18T19:00:00.000Z,0.04400000000000002,244.46549999999996,0.19549999999999995,49.983999999999995
BR03,2019-12-18T20:00:00.000Z,0.039000000000000014,244.877,0.16949999999999998,49.9765
BR03,2019-12-18T21:00:00.000Z,0.03500000000000001,245.33199999999997,0.15300000000000005,50.01800000000001
BR03,2019-12-18T22:00:00.000Z,0.029000000000000012,248.706,0.12000000000000006,50.02599999999999
BR03,2019-12-18T23:00:00.000Z,0.02800000000000001,251.40600000000003,0.12000000000000006,50.028499999999994
BR03,2019-12-20T00:00:00.000Z,0.10900000000000003,253.26099999999997,0.43599999999999994,50.003
BR03,2019-12-20T01:00:00.000Z,0.10500000000000002,254.01350000000002,0.42000000000000004,50.003


In [0]:
# ---------------------------------------------------------
# 2. Create synthetic 168-hour history for NEW users
# ---------------------------------------------------------

existing_meter_ids = [
    row["meter_id"]
    for row in historical_registered_df
    .select("meter_id")
    .distinct()
    .collect()
]

new_users = [
    user
    for user in registered_users
    if user["meter_id"] not in existing_meter_ids
]

synthetic_history = []

current_time = datetime.now().replace(
    minute=0,
    second=0,
    microsecond=0
)

for user in new_users:

    for hour_back in range(168, 0, -1):

        timestamp = (
            current_time
            - timedelta(hours=hour_back)
        )

        baseline_energy = user["baseline_energy_3min"] * 20

        # Small realistic variation
        energy = baseline_energy * random.uniform(
            0.85,
            1.15
        )

        voltage = user["baseline_voltage"] + random.uniform(
            -2,
            2
        )

        current = user["baseline_current"] * random.uniform(
            0.85,
            1.15
        )

        frequency = user["baseline_frequency"] + random.uniform(
            -0.03,
            0.03
        )

        synthetic_history.append(
            (
                user["meter_id"],
                timestamp,
                float(energy),
                float(voltage),
                float(current),
                float(frequency)
            )
        )

synthetic_history_df = spark.createDataFrame(
    synthetic_history,
    [
        "meter_id",
        "hour_timestamp",
        "energy_kwh",
        "avg_voltage",
        "avg_current",
        "frequency"
    ]
)

print(
    "New users:",
    len(new_users)
)

print(
    "Synthetic history rows:",
    synthetic_history_df.count()
)

display(synthetic_history_df.limit(10))

New users: 1
Synthetic history rows: 168


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency
BR100,2026-09-13T07:00:00.000Z,3.2412267944564084,231.06686185386386,3.940301540771538,49.97046807096916
BR100,2026-09-13T08:00:00.000Z,2.8326870515866673,231.62825248559824,5.1539806403214214,49.99130496319506
BR100,2026-09-13T09:00:00.000Z,3.1866520462055528,231.9989058482702,4.691062963562775,49.993685800937996
BR100,2026-09-13T10:00:00.000Z,2.7977215929104178,228.2335044857184,4.255288389944741,49.9796932701334
BR100,2026-09-13T11:00:00.000Z,2.7769736961414155,230.44886264309787,4.52088146215204,50.00245854353639
BR100,2026-09-13T12:00:00.000Z,2.7321783252570824,229.3429727490158,4.377555578524447,49.98078471856281
BR100,2026-09-13T13:00:00.000Z,2.7113336189249315,229.58543696085025,4.739961350307614,49.9719339378794
BR100,2026-09-13T14:00:00.000Z,3.1205381364113802,231.11027885323125,4.762110839109512,49.99537448860897
BR100,2026-09-13T15:00:00.000Z,2.6304701995999755,229.61967384984297,5.129928595752052,50.02439408770514
BR100,2026-09-13T16:00:00.000Z,2.9526393396982753,231.18160507569493,4.373745602744531,49.99136097712635


In [0]:
# ---------------------------------------------------------
# 3. Combine historical/synthetic seed with live hourly data
# ---------------------------------------------------------

forecast_history_df = (
    historical_seed_df
    .select(
        "meter_id",
        "hour_timestamp",
        "energy_kwh",
        "avg_voltage",
        "avg_current",
        "frequency"
    )
    .unionByName(
        synthetic_history_df
    )
)

# Add current live hourly data
forecast_history_df = (
    forecast_history_df
    .unionByName(
        live_hourly_df.select(
            "meter_id",
            "hour_timestamp",
            "energy_kwh",
            "avg_voltage",
            "avg_current",
            "frequency"
        )
    )
    .dropDuplicates(
        ["meter_id", "hour_timestamp"]
    )
    .orderBy(
        "meter_id",
        "hour_timestamp"
    )
)

print("✅ Forecasting history created!")

display(forecast_history_df)

✅ Forecasting history created!


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency
BR03,2019-12-18T16:00:00.000Z,0.04600000000000001,243.68900000000002,0.20600000000000002,50.01050000000001
BR03,2019-12-18T17:00:00.000Z,0.04900000000000001,243.52549999999997,0.21750000000000008,50.020500000000006
BR03,2019-12-18T18:00:00.000Z,0.05500000000000001,243.24949999999998,0.27050000000000013,50.031499999999994
BR03,2019-12-18T19:00:00.000Z,0.04400000000000002,244.46549999999996,0.19549999999999995,49.983999999999995
BR03,2019-12-18T20:00:00.000Z,0.039000000000000014,244.877,0.16949999999999998,49.9765
BR03,2019-12-18T21:00:00.000Z,0.03500000000000001,245.33199999999997,0.15300000000000005,50.01800000000001
BR03,2019-12-18T22:00:00.000Z,0.029000000000000012,248.706,0.12000000000000006,50.02599999999999
BR03,2019-12-18T23:00:00.000Z,0.02800000000000001,251.40600000000003,0.12000000000000006,50.028499999999994
BR03,2019-12-20T00:00:00.000Z,0.10900000000000003,253.26099999999997,0.43599999999999994,50.003
BR03,2019-12-20T01:00:00.000Z,0.10500000000000002,254.01350000000002,0.42000000000000004,50.003


In [0]:
forecast_live_window = (
    Window
    .partitionBy("meter_id")
    .orderBy("hour_timestamp")
)

forecast_live_df = (
    forecast_history_df

    .withColumn(
        "hour",
        F.hour("hour_timestamp")
    )

    .withColumn(
        "day",
        F.dayofmonth("hour_timestamp")
    )

    .withColumn(
        "month",
        F.month("hour_timestamp")
    )

    .withColumn(
        "day_of_week",
        F.dayofweek("hour_timestamp")
    )

    .withColumn(
        "is_weekend",
        F.when(
            F.dayofweek("hour_timestamp").isin([1, 7]),
            1
        ).otherwise(0)
    )

    .withColumn(
        "energy_lag_1",
        F.lag("energy_kwh", 1)
        .over(forecast_live_window)
    )

    .withColumn(
        "energy_lag_2",
        F.lag("energy_kwh", 2)
        .over(forecast_live_window)
    )

    .withColumn(
        "energy_lag_24",
        F.lag("energy_kwh", 24)
        .over(forecast_live_window)
    )

    .withColumn(
        "energy_lag_168",
        F.lag("energy_kwh", 168)
        .over(forecast_live_window)
    )
)

print("✅ Live lag features created!")

display(
    forecast_live_df
    .orderBy(F.col("hour_timestamp").desc())
    .limit(10)
)

✅ Live lag features created!


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,day_of_week,is_weekend,energy_lag_1,energy_lag_2,energy_lag_24,energy_lag_168
BR03,2026-09-20T07:00:00.000Z,0.0652,243.77,1.4,50.0,7,20,9,1,1,0.1422,0.27899999999999997,0.5262,0.048000000000000015
BR100,2026-09-20T06:00:00.000Z,2.7444321411101056,231.8927399804689,4.992503571044735,49.99001020954785,6,20,9,1,1,2.81587868659331,3.3757675404132015,2.562124373468501,1.4266999999999999
BR100,2026-09-20T05:00:00.000Z,2.81587868659331,231.6030903894873,4.428390385799917,50.00354637496878,5,20,9,1,1,3.3757675404132015,3.4123329448230875,3.015964878781005,3.0284
BR100,2026-09-20T04:00:00.000Z,3.3757675404132015,229.90812370959335,4.270795765741355,50.026238784582134,4,20,9,1,1,3.4123329448230875,2.924170893657661,2.8831878711807115,3.0671999999999997
BR100,2026-09-20T03:00:00.000Z,3.4123329448230875,229.29492221702793,5.132012023280053,49.978137706698284,3,20,9,1,1,2.924170893657661,3.0011662467686926,2.9501463917613724,3.0593000000000004
BR100,2026-09-20T02:00:00.000Z,2.924170893657661,231.09582898564426,4.295875573316081,49.984985515492156,2,20,9,1,1,3.0011662467686926,3.3925103353567048,3.185885053049095,2.8647
BR100,2026-09-20T01:00:00.000Z,3.0011662467686926,231.4530018011623,4.588029038041429,49.9944868904465,1,20,9,1,1,3.3925103353567048,2.777446598986149,2.942412362811364,2.9618
BR100,2026-09-20T00:00:00.000Z,3.3925103353567048,229.9709493627341,3.901031017612254,50.001531532712875,0,20,9,1,1,2.777446598986149,2.6113787792900207,3.3656726549717435,2.9045
BR100,2026-09-19T23:00:00.000Z,2.777446598986149,231.7483104440275,4.826517722030082,50.012821893318026,23,19,9,7,1,2.6113787792900207,3.199286195187131,2.712423256695608,2.8826
BR100,2026-09-19T22:00:00.000Z,2.6113787792900207,231.9317346696638,4.490073436551497,49.97933576744646,22,19,9,7,1,3.199286195187131,3.225458115600021,2.929340146260924,3.0209


In [0]:
rolling_live_window = (
    Window
    .partitionBy("meter_id")
    .orderBy("hour_timestamp")
    .rowsBetween(-24, -1)
)

forecast_live_df = (
    forecast_live_df
    .withColumn(
        "rolling_avg_24h",
        F.avg("energy_kwh").over(
            rolling_live_window
        )
    )
)

print("✅ Rolling 24-hour average created!")

display(
    forecast_live_df
    .orderBy(F.col("hour_timestamp").desc())
    .limit(10)
)

✅ Rolling 24-hour average created!


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,day_of_week,is_weekend,energy_lag_1,energy_lag_2,energy_lag_24,energy_lag_168,rolling_avg_24h
BR03,2026-09-20T07:00:00.000Z,0.0652,243.77,1.4,50.0,7,20,9,1,1,0.1422,0.27899999999999997,0.5262,0.048000000000000015,0.28322916666666664
BR100,2026-09-20T06:00:00.000Z,2.7444321411101056,231.8927399804689,4.992503571044735,49.99001020954785,6,20,9,1,1,2.81587868659331,3.3757675404132015,2.562124373468501,1.4266999999999999,3.027990546495475
BR100,2026-09-20T05:00:00.000Z,2.81587868659331,231.6030903894873,4.428390385799917,50.00354637496878,5,20,9,1,1,3.3757675404132015,3.4123329448230875,3.015964878781005,3.0284,3.036327471169962
BR100,2026-09-20T04:00:00.000Z,3.3757675404132015,229.90812370959335,4.270795765741355,50.026238784582134,4,20,9,1,1,3.4123329448230875,2.924170893657661,2.8831878711807115,3.0671999999999997,3.0158033182852755
BR100,2026-09-20T03:00:00.000Z,3.4123329448230875,229.29492221702793,5.132012023280053,49.978137706698284,3,20,9,1,1,2.924170893657661,3.0011662467686926,2.9501463917613724,3.0593000000000004,2.996545545241037
BR100,2026-09-20T02:00:00.000Z,2.924170893657661,231.09582898564426,4.295875573316081,49.984985515492156,2,20,9,1,1,3.0011662467686926,3.3925103353567048,3.185885053049095,2.8647,3.0074503018823466
BR100,2026-09-20T01:00:00.000Z,3.0011662467686926,231.4530018011623,4.588029038041429,49.9944868904465,1,20,9,1,1,3.3925103353567048,2.777446598986149,2.942412362811364,2.9618,3.0050022233841247
BR100,2026-09-20T00:00:00.000Z,3.3925103353567048,229.9709493627341,3.901031017612254,50.001531532712875,0,20,9,1,1,2.777446598986149,2.6113787792900207,3.3656726549717435,2.9045,3.0038839867014175
BR100,2026-09-19T23:00:00.000Z,2.777446598986149,231.7483104440275,4.826517722030082,50.012821893318026,23,19,9,7,1,2.6113787792900207,3.199286195187131,2.712423256695608,2.8826,3.0011746807726456
BR100,2026-09-19T22:00:00.000Z,2.6113787792900207,231.9317346696638,4.490073436551497,49.97933576744646,22,19,9,7,1,3.199286195187131,3.225458115600021,2.929340146260924,3.0209,3.0144230710630997


In [0]:
latest_forecast_window = (
    Window
    .partitionBy("meter_id")
    .orderBy(
        F.col("hour_timestamp").desc()
    )
)

latest_forecast_df = (
    forecast_live_df
    .withColumn(
        "row_num",
        F.row_number().over(
            latest_forecast_window
        )
    )
    .filter(
        F.col("row_num") == 1
    )
    .drop("row_num")
)

print("✅ Latest forecasting row selected for each user!")

display(latest_forecast_df)

✅ Latest forecasting row selected for each user!


meter_id,hour_timestamp,energy_kwh,avg_voltage,avg_current,frequency,hour,day,month,day_of_week,is_weekend,energy_lag_1,energy_lag_2,energy_lag_24,energy_lag_168,rolling_avg_24h
BR03,2026-09-20T07:00:00.000Z,0.0652,243.77,1.4,50.0,7,20,9,1,1,0.1422,0.27899999999999997,0.5262,0.048000000000000015,0.28322916666666664
BR100,2026-09-20T06:00:00.000Z,2.7444321411101056,231.8927399804689,4.992503571044735,49.99001020954785,6,20,9,1,1,2.81587868659331,3.3757675404132015,2.562124373468501,1.4266999999999999,3.027990546495475


In [0]:
print("Checking forecasting features...\n")

for feature in feature_cols:
    if feature in latest_forecast_df.columns:
        print("✅", feature)
    else:
        print("❌ MISSING:", feature)

Checking forecasting features...

✅ hour
✅ day
✅ month
✅ day_of_week
✅ is_weekend
✅ avg_voltage
✅ avg_current
✅ frequency
✅ energy_lag_1
✅ energy_lag_2
✅ energy_lag_24
✅ energy_lag_168
✅ rolling_avg_24h


In [0]:
live_forecast_pd = latest_forecast_df.select(
    "meter_id",
    "hour_timestamp",
    *feature_cols
).toPandas()

print("✅ Live forecasting data prepared!")

print(
    "Users available:",
    len(live_forecast_pd)
)

display(live_forecast_pd)

✅ Live forecasting data prepared!
Users available: 2


meter_id,hour_timestamp,hour,day,month,day_of_week,is_weekend,avg_voltage,avg_current,frequency,energy_lag_1,energy_lag_2,energy_lag_24,energy_lag_168,rolling_avg_24h
BR03,2026-09-20T07:00:00.000Z,7,20,9,1,1,243.77,1.4,50.0,0.1422,0.27899999999999997,0.5262,0.048000000000000015,0.28322916666666664
BR100,2026-09-20T06:00:00.000Z,6,20,9,1,1,231.8927399804689,4.992503571044735,49.99001020954785,2.81587868659331,3.3757675404132015,2.562124373468501,1.4266999999999999,3.027990546495475


In [0]:
print("Missing values in live forecasting features:")
print(
    live_forecast_pd[feature_cols]
    .isnull()
    .sum()
)

Missing values in live forecasting features:
hour               0
day                0
month              0
day_of_week        0
is_weekend         0
avg_voltage        0
avg_current        0
frequency          0
energy_lag_1       0
energy_lag_2       0
energy_lag_24      0
energy_lag_168     0
rolling_avg_24h    0
dtype: int64


In [0]:
X_live = live_forecast_pd[feature_cols]

live_forecast_pd["xgb_next_hour_kwh"] = (
    xgb_live_model.predict(X_live)
)

print("✅ XGBoost live forecast generated!")

display(
    live_forecast_pd[
        [
            "meter_id",
            "hour_timestamp",
            "xgb_next_hour_kwh"
        ]
    ]
)

✅ XGBoost live forecast generated!


meter_id,hour_timestamp,xgb_next_hour_kwh
BR03,2026-09-20T07:00:00.000Z,0.2912011
BR100,2026-09-20T06:00:00.000Z,0.9758431


In [0]:
live_forecast_pd["rf_next_hour_kwh"] = (
    rf_live_model.predict(X_live)
)

print("✅ Random Forest live forecast generated!")

display(
    live_forecast_pd[
        [
            "meter_id",
            "hour_timestamp",
            "xgb_next_hour_kwh",
            "rf_next_hour_kwh"
        ]
    ]
)

✅ Random Forest live forecast generated!


meter_id,hour_timestamp,xgb_next_hour_kwh,rf_next_hour_kwh
BR03,2026-09-20T07:00:00.000Z,0.2912011,0.3041175419656867
BR100,2026-09-20T06:00:00.000Z,0.9758431,1.2850252979773304


In [0]:
live_forecast_pd["predicted_next_hour_kwh"] = (
    live_forecast_pd["xgb_next_hour_kwh"]
)

print("✅ Primary live forecast created!")

display(
    live_forecast_pd[
        [
            "meter_id",
            "xgb_next_hour_kwh",
            "rf_next_hour_kwh",
            "predicted_next_hour_kwh"
        ]
    ]
)

✅ Primary live forecast created!


meter_id,xgb_next_hour_kwh,rf_next_hour_kwh,predicted_next_hour_kwh
BR03,0.2912011,0.3041175419656867,0.2912011
BR100,0.9758431,1.2850252979773304,0.9758431


In [0]:
# Prepare live forecasting features

X_live = live_forecast_pd[feature_cols]

# Generate next-hour prediction
live_forecast_pd["xgb_next_hour_kwh"] = (
    xgb_live_model.predict(X_live)
)

print("✅ XGBoost live forecast generated!")

display(
    live_forecast_pd[
        [
            "meter_id",
            "hour_timestamp",
            "xgb_next_hour_kwh"
        ]
    ]
)

✅ XGBoost live forecast generated!


meter_id,hour_timestamp,xgb_next_hour_kwh
BR03,2026-09-20T07:00:00.000Z,0.2912011
BR100,2026-09-20T06:00:00.000Z,0.9758431


In [0]:
# Generate Random Forest next-hour prediction

live_forecast_pd["rf_next_hour_kwh"] = (
    rf_live_model.predict(X_live)
)

print("✅ Random Forest live forecast generated!")

display(
    live_forecast_pd[
        [
            "meter_id",
            "hour_timestamp",
            "xgb_next_hour_kwh",
            "rf_next_hour_kwh"
        ]
    ]
)

✅ Random Forest live forecast generated!


meter_id,hour_timestamp,xgb_next_hour_kwh,rf_next_hour_kwh
BR03,2026-09-20T07:00:00.000Z,0.2912011,0.3041175419656867
BR100,2026-09-20T06:00:00.000Z,0.9758431,1.2850252979773304


In [0]:
live_forecast_spark_df = spark.createDataFrame(
    live_forecast_pd[
        [
            "meter_id",
            "hour_timestamp",
            "xgb_next_hour_kwh",
            "rf_next_hour_kwh",
            "predicted_next_hour_kwh"
        ]
    ]
)

print("✅ Live forecast converted to Spark DataFrame!")

display(live_forecast_spark_df)

✅ Live forecast converted to Spark DataFrame!


meter_id,hour_timestamp,xgb_next_hour_kwh,rf_next_hour_kwh,predicted_next_hour_kwh
BR03,2026-09-20T07:00:00.000Z,0.2912011,0.3041175419656867,0.2912011
BR100,2026-09-20T06:00:00.000Z,0.9758431,1.2850252979773304,0.9758431


In [0]:
dashboard_live_df = (
    dashboard_live_df
    .join(
        live_forecast_spark_df,
        on="meter_id",
        how="left"
    )
)

display(dashboard_live_df)

meter_id,user_type,home_type,occupants,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,today_energy_kwh,monthly_target_kwh,monthly_budget,projected_monthly_usage,projected_bill,projected_usage_difference,budget_difference,guardian_status,budget_status,anomaly_count,anomaly_status,hour_timestamp,xgb_next_hour_kwh,rf_next_hour_kwh,predicted_next_hour_kwh
BR03,Existing User,Apartment,3,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,0.0652,200.0,1500.0,0.0978,0.2934,-199.9022,1499.7066,ON TRACK,WITHIN BUDGET,0,NORMAL,2026-09-20T07:00:00.000Z,0.2912011,0.3041175419656867,0.2912011
BR100,New User,House,4,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,0.0,250.0,1800.0,0.0,0.0,-250.0,1800.0,ON TRACK,WITHIN BUDGET,11,ANOMALY DETECTED,2026-09-20T06:00:00.000Z,0.9758431,1.2850252979773304,0.9758431


In [0]:
dashboard_live_df = (
    dashboard_live_df

    .withColumn(
        "forecast_status",
        F.when(
            F.col("predicted_next_hour_kwh")
            > F.col("monthly_target_kwh") / 30 / 24,
            "HIGH EXPECTED USAGE"
        )
        .otherwise("NORMAL EXPECTED USAGE")
    )
)

display(dashboard_live_df)

meter_id,user_type,home_type,occupants,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,today_energy_kwh,monthly_target_kwh,monthly_budget,projected_monthly_usage,projected_bill,projected_usage_difference,budget_difference,guardian_status,budget_status,anomaly_count,anomaly_status,hour_timestamp,xgb_next_hour_kwh,rf_next_hour_kwh,predicted_next_hour_kwh,forecast_status
BR03,Existing User,Apartment,3,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,0.0652,200.0,1500.0,0.0978,0.2934,-199.9022,1499.7066,ON TRACK,WITHIN BUDGET,0,NORMAL,2026-09-20T07:00:00.000Z,0.2912011,0.3041175419656867,0.2912011,HIGH EXPECTED USAGE
BR100,New User,House,4,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,0.0,250.0,1800.0,0.0,0.0,-250.0,1800.0,ON TRACK,WITHIN BUDGET,11,ANOMALY DETECTED,2026-09-20T06:00:00.000Z,0.9758431,1.2850252979773304,0.9758431,HIGH EXPECTED USAGE


In [0]:
final_dashboard_df = dashboard_live_df.select(
    "meter_id",
    "user_type",
    "home_type",
    "occupants",

    # Live information
    "last_update",
    "latest_energy_kwh",
    "latest_voltage",
    "latest_current",
    "latest_frequency",

    # Consumption
    "today_energy_kwh",

    # User planning
    "monthly_target_kwh",
    "monthly_budget",

    # Budget Guardian
    "projected_monthly_usage",
    "projected_bill",
    "projected_usage_difference",
    "budget_difference",
    "guardian_status",
    "budget_status",

    # Anomaly
    "anomaly_count",
    "anomaly_status",

    # Forecasting
    "xgb_next_hour_kwh",
    "rf_next_hour_kwh",
    "predicted_next_hour_kwh",
    "forecast_status"
)

display(final_dashboard_df)

meter_id,user_type,home_type,occupants,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,today_energy_kwh,monthly_target_kwh,monthly_budget,projected_monthly_usage,projected_bill,projected_usage_difference,budget_difference,guardian_status,budget_status,anomaly_count,anomaly_status,xgb_next_hour_kwh,rf_next_hour_kwh,predicted_next_hour_kwh,forecast_status
BR03,Existing User,Apartment,3,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,0.0652,200.0,1500.0,0.0978,0.2934,-199.9022,1499.7066,ON TRACK,WITHIN BUDGET,0,NORMAL,0.2912011,0.3041175419656867,0.2912011,HIGH EXPECTED USAGE
BR100,New User,House,4,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,0.0,250.0,1800.0,0.0,0.0,-250.0,1800.0,ON TRACK,WITHIN BUDGET,11,ANOMALY DETECTED,0.9758431,1.2850252979773304,0.9758431,HIGH EXPECTED USAGE


In [0]:
final_dashboard_table = "workspace.default.smart_energy_dashboard"

(
    final_dashboard_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(final_dashboard_table)
)

print("✅ FINAL DASHBOARD TABLE UPDATED!")

✅ FINAL DASHBOARD TABLE UPDATED!


In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {final_dashboard_table}
        ORDER BY last_update DESC
    """)
)

meter_id,user_type,home_type,occupants,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,today_energy_kwh,monthly_target_kwh,monthly_budget,projected_monthly_usage,projected_bill,projected_usage_difference,budget_difference,guardian_status,budget_status,anomaly_count,anomaly_status,xgb_next_hour_kwh,rf_next_hour_kwh,predicted_next_hour_kwh,forecast_status
BR03,Existing User,Apartment,3,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,0.0652,200.0,1500.0,0.0978,0.2934,-199.9022,1499.7066,ON TRACK,WITHIN BUDGET,0,NORMAL,0.2912011,0.3041175419656867,0.2912011,HIGH EXPECTED USAGE
BR100,New User,House,4,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,0.0,250.0,1800.0,0.0,0.0,-250.0,1800.0,ON TRACK,WITHIN BUDGET,11,ANOMALY DETECTED,0.9758431,1.2850252979773304,0.9758431,HIGH EXPECTED USAGE


In [0]:
display(
    spark.table(final_dashboard_table)
    .orderBy(F.col("last_update").desc())
)

meter_id,user_type,home_type,occupants,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,today_energy_kwh,monthly_target_kwh,monthly_budget,projected_monthly_usage,projected_bill,projected_usage_difference,budget_difference,guardian_status,budget_status,anomaly_count,anomaly_status,xgb_next_hour_kwh,rf_next_hour_kwh,predicted_next_hour_kwh,forecast_status
BR03,Existing User,Apartment,3,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,0.0652,200.0,1500.0,0.0978,0.2934,-199.9022,1499.7066,ON TRACK,WITHIN BUDGET,0,NORMAL,0.2912011,0.3041175419656867,0.2912011,HIGH EXPECTED USAGE
BR100,New User,House,4,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,0.0,250.0,1800.0,0.0,0.0,-250.0,1800.0,ON TRACK,WITHIN BUDGET,11,ANOMALY DETECTED,0.9758431,1.2850252979773304,0.9758431,HIGH EXPECTED USAGE


In [0]:
dashboard_df = spark.table(final_dashboard_table)

display(
    dashboard_df.select(
        "meter_id",
        "user_type",
        "home_type",
        "occupants",
        "last_update",
        "latest_energy_kwh",
        "latest_voltage",
        "latest_current",
        "latest_frequency",
        "today_energy_kwh",
        "monthly_target_kwh",
        "monthly_budget",
        "projected_monthly_usage",
        "projected_bill",
        "guardian_status",
        "budget_status",
        "anomaly_count",
        "anomaly_status",
        "xgb_next_hour_kwh",
        "rf_next_hour_kwh",
        "predicted_next_hour_kwh",
        "forecast_status"
    )
)

meter_id,user_type,home_type,occupants,last_update,latest_energy_kwh,latest_voltage,latest_current,latest_frequency,today_energy_kwh,monthly_target_kwh,monthly_budget,projected_monthly_usage,projected_bill,guardian_status,budget_status,anomaly_count,anomaly_status,xgb_next_hour_kwh,rf_next_hour_kwh,predicted_next_hour_kwh,forecast_status
BR03,Existing User,Apartment,3,2026-09-20T07:10:31.559Z,0.0652,243.77,1.4,50.0,0.0652,200.0,1500.0,0.0978,0.2934,ON TRACK,WITHIN BUDGET,0,NORMAL,0.2912011,0.3041175419656867,0.2912011,HIGH EXPECTED USAGE
BR100,New User,House,4,2026-09-02T17:15:00.000Z,0.7515,228.66,4.82,49.97,0.0,250.0,1800.0,0.0,0.0,ON TRACK,WITHIN BUDGET,11,ANOMALY DETECTED,0.9758431,1.2850252979773304,0.9758431,HIGH EXPECTED USAGE


In [0]:
from pyspark.sql import functions as F

recommendation_df = (
    dashboard_live_df
    .withColumn(
        "energy_recommendation",
        F.when(
            F.col("guardian_status") == "USAGE TARGET RISK",
            "Reduce daily electricity usage to stay within the monthly target."
        )
        .when(
            F.col("budget_status") == "BUDGET RISK",
            "Reduce energy consumption to avoid exceeding the monthly budget."
        )
        .when(
            F.col("anomaly_status") == "ANOMALY DETECTED",
            "Unusual energy usage detected. Check high-consumption appliances."
        )
        .otherwise(
            "Energy usage is normal. Continue your current energy-saving habits."
        )
    )
)

display(
    recommendation_df.select(
        "meter_id",
        "guardian_status",
        "budget_status",
        "anomaly_status",
        "energy_recommendation"
    )
)

meter_id,guardian_status,budget_status,anomaly_status,energy_recommendation
BR03,ON TRACK,WITHIN BUDGET,NORMAL,Energy usage is normal. Continue your current energy-saving habits.
BR100,ON TRACK,WITHIN BUDGET,ANOMALY DETECTED,Unusual energy usage detected. Check high-consumption appliances.


In [0]:
from pyspark.sql import functions as F

recommendation_df = (
    fresh_dashboard
    .withColumn(
        "energy_recommendation",
        F.when(
            F.col("budget_status") == "BUDGET RISK",
            "Projected bill exceeds the monthly budget. Reduce high-load appliance usage and monitor peak consumption periods."
        )
        .when(
            F.col("guardian_status") == "USAGE TARGET RISK",
            "Projected consumption exceeds the monthly target. Consider reducing daily usage and reviewing high-consumption appliances."
        )
        .when(
            F.col("anomaly_status") == "ANOMALY DETECTED",
            "Unusual consumption detected. Review recent energy readings and check high-consumption appliances."
        )
        .otherwise(
            "Consumption is within the defined target and budget. Continue monitoring usage and maintain current energy-saving practices."
        )
    )
)

display(
    recommendation_df.select(
        "meter_id",
        "guardian_status",
        "budget_status",
        "anomaly_status",
        "energy_recommendation"
    )
)

meter_id,guardian_status,budget_status,anomaly_status,energy_recommendation
BR03,ON TRACK,WITHIN BUDGET,NORMAL,Consumption is within the defined target and budget. Continue monitoring usage and maintain current energy-saving practices.
BR100,USAGE TARGET RISK,BUDGET RISK,ANOMALY DETECTED,Projected bill exceeds the monthly budget. Reduce high-load appliance usage and monitor peak consumption periods.


In [0]:
from pyspark.sql import functions as F

# Keep only the recommendation text from recommendation_df
recommendation_only = (
    recommendation_df
    .select(
        "meter_id",
        "energy_recommendation"
    )
)

# Add recommendation to the main dashboard dataset
final_dashboard_df = (
    fresh_dashboard
    .drop(
        "energy_recommendation"
    )
    .join(
        recommendation_only,
        on="meter_id",
        how="left"
    )
)

# Save updated dashboard table
(
    final_dashboard_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.smart_energy_dashboard")
)

print("✅ Recommendation added to main dashboard table!")

display(
    final_dashboard_df.select(
        "meter_id",
        "energy_recommendation"
    )
)

✅ Recommendation added to main dashboard table!


meter_id,energy_recommendation
BR03,Consumption is within the defined target and budget. Continue monitoring usage and maintain current energy-saving practices.
BR100,Projected bill exceeds the monthly budget. Reduce high-load appliance usage and monitor peak consumption periods.


In [0]:
from pyspark.sql import functions as F

display(
    live_raw_df
    .groupBy("meter_id")
    .agg(
        F.min("timestamp").alias("min_timestamp"),
        F.max("timestamp").alias("max_timestamp"),
        F.count("*").alias("total_readings")
    )
    .orderBy("meter_id")
)

meter_id,min_timestamp,max_timestamp,total_readings
BR03,2026-09-01T00:00:00.000Z,2026-09-20T07:10:31.559Z,277
BR100,2026-09-01T00:00:00.000Z,2026-09-02T17:15:00.000Z,274


In [0]:
spark.sql(f"""
DELETE FROM {live_table_name}
WHERE meter_id = 'BR03'
  AND timestamp >= '2026-09-03'
""")

print("✅ Stale BR03 records removed.")

✅ Stale BR03 records removed.


In [0]:
display(
    spark.table(live_table_name)
    .groupBy("meter_id")
    .agg(
        F.min("timestamp").alias("min_timestamp"),
        F.max("timestamp").alias("max_timestamp"),
        F.count("*").alias("total_readings")
    )
    .orderBy("meter_id")
)

meter_id,min_timestamp,max_timestamp,total_readings
BR03,2026-09-01T00:00:00.000Z,2026-09-02T17:15:00.000Z,276
BR100,2026-09-01T00:00:00.000Z,2026-09-02T17:15:00.000Z,274


In [0]:
live_raw_df = spark.table(live_table_name)

print("Live rows:", live_raw_df.count())

display(
    live_raw_df
    .groupBy("meter_id")
    .agg(
        F.sum("energy_kwh").alias("total_simulated_energy"),
        F.min("timestamp").alias("start_time"),
        F.max("timestamp").alias("latest_time")
    )
)

Live rows: 550


meter_id,total_simulated_energy,start_time,latest_time
BR03,18.22819999999999,2026-09-01T00:00:00.000Z,2026-09-02T17:15:00.000Z
BR100,210.16610000000006,2026-09-01T00:00:00.000Z,2026-09-02T17:15:00.000Z


In [0]:
from pyspark.sql import functions as F

# Refresh live data
live_raw_df = spark.table(live_table_name)

# Latest simulated date
simulation_date = (
    live_raw_df
    .select(F.to_date(F.max("timestamp")).alias("simulation_date"))
    .collect()[0]["simulation_date"]
)

current_day = simulation_date.day

print("Simulation date:", simulation_date)
print("Simulation day:", current_day)

# Calculate usage for the latest simulated day
fresh_daily_usage = (
    live_raw_df
    .filter(F.to_date("timestamp") == F.lit(simulation_date))
    .groupBy("meter_id")
    .agg(
        F.sum("energy_kwh").alias("today_energy_kwh")
    )
)

# Load existing dashboard
dashboard_current = spark.table(
    "workspace.default.smart_energy_dashboard"
)

# Recalculate bill-related metrics
fresh_dashboard = (
    dashboard_current
    .drop(
        "today_energy_kwh",
        "projected_monthly_usage",
        "projected_bill",
        "projected_usage_difference",
        "budget_difference",
        "guardian_status",
        "budget_status"
    )
    .join(
        fresh_daily_usage,
        on="meter_id",
        how="left"
    )
    .fillna({"today_energy_kwh": 0.0})
    .withColumn(
        "projected_monthly_usage",
        (F.col("today_energy_kwh") / F.lit(current_day)) * 30
    )
    .withColumn(
        "projected_usage_difference",
        F.col("projected_monthly_usage")
        - F.col("monthly_target_kwh")
    )
    .withColumn(
        "projected_bill",
        F.when(
            F.col("projected_monthly_usage") <= 50,
            F.col("projected_monthly_usage") * 3
        )
        .when(
            F.col("projected_monthly_usage") <= 150,
            (50 * 3)
            + ((F.col("projected_monthly_usage") - 50) * 5)
        )
        .when(
            F.col("projected_monthly_usage") <= 250,
            (50 * 3)
            + (100 * 5)
            + ((F.col("projected_monthly_usage") - 150) * 7)
        )
        .otherwise(
            (50 * 3)
            + (100 * 5)
            + (100 * 7)
            + ((F.col("projected_monthly_usage") - 250) * 10)
        )
    )
    .withColumn(
        "budget_difference",
        F.col("monthly_budget") - F.col("projected_bill")
    )
    .withColumn(
        "guardian_status",
        F.when(
            F.col("projected_monthly_usage")
            > F.col("monthly_target_kwh"),
            "USAGE TARGET RISK"
        ).otherwise("ON TRACK")
    )
    .withColumn(
        "budget_status",
        F.when(
            F.col("projected_bill")
            > F.col("monthly_budget"),
            "BUDGET RISK"
        ).otherwise("WITHIN BUDGET")
    )
)

# Save updated dashboard table
(
    fresh_dashboard
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.smart_energy_dashboard")
)

print("✅ Dashboard billing metrics updated!")

display(
    fresh_dashboard.select(
        "meter_id",
        "today_energy_kwh",
        "projected_monthly_usage",
        "projected_bill",
        "monthly_target_kwh",
        "monthly_budget",
        "guardian_status",
        "budget_status"
    )
)

Simulation date: 2026-09-02
Simulation day: 2
✅ Dashboard billing metrics updated!


meter_id,today_energy_kwh,projected_monthly_usage,projected_bill,monthly_target_kwh,monthly_budget,guardian_status,budget_status
BR03,4.4735000000000005,67.1025,235.51250000000005,200.0,1500.0,ON TRACK,WITHIN BUDGET
BR100,51.60849999999998,774.1274999999997,6591.274999999997,250.0,1800.0,USAGE TARGET RISK,BUDGET RISK
